# Notebook to analize databased format from WP6

In [28]:
from pathlib import Path
import pandas as pd
import zipfile

In [14]:
FOLDER = Path(".")

## Investigate the EXCEL files 

In [17]:
for path in FOLDER.iterdir():
    if path.suffix.lower() in [".xlsx", ".xls"]:
        print(f"\nEXCEL: {path.name}")
        xls = pd.ExcelFile(path)
        print("Sheets:", xls.sheet_names)

    elif path.suffix.lower() == ".csv":
        print(f"\nCSV: {path.name}")
        df = pd.read_csv(path, sep=None, engine="python", nrows=5)
        print("Columns:", list(df.columns))
        print(df.head())


EXCEL: 24ACE-FH-EARLY_DHM_v105_20260424.xls
Sheets: ['Data Handling Manual FH-EARLY']

EXCEL: 24ACE-FH-Early_import_DATA_20251015.xlsx
Sheets: ['Notice', 'INCLUSION', 'ML', 'Gene refseq', 'Family cardiovascular events']


## Inspect `Data Handling Manual FH-EARLY`
We have: 
* `Référence` that is the variable name
* `Type` that is the variable type
* `Format` that is the codes/rules
* `Intitulé` that is the label

In [22]:
file = Path("24ACE-FH-EARLY_DHM_v105_20260424.xls")

df = pd.read_excel(file, sheet_name="Data Handling Manual FH-EARLY")

print("Columns:")
for col in df.columns:
    print(f"- {col}")

print("\nFirst 10 rows:")
print(df)

Columns:
- Id
- Référence
- Saisie
- Type
- Format
- Intitulé

First 10 rows:
                                                   Id           Référence  \
0                                                 NaN                 NaN   
1    INCLUSION (INC)   :  Patient identification (ID)                 NaN   
2                                           ID102S2V9         PROJECTNAME   
3                                                  V5                 SEX   
4                                          ID102S2V10             BRTHDAT   
..                                                ...                 ...   
232                                       ID102S2V199            DATA_CNI   
233                                       ID102S2V172            DATA_DT1   
234                                       ID102S2V173           DATA_NUM2   
235                                       ID102S2V183  RFINSTMOINS1MOISDT   
236                                       ID102S2V203         DATA_INCLDT  

### Investigate variables

In [23]:
variables = df[df["Référence"].notna()].copy()

variables = variables[["Id", "Référence", "Saisie", "Type", "Format", "Intitulé"]]

variables.to_csv("data_dictionary_clean.csv", index=False)

print(variables.head(20))
print(f"\nSaved {len(variables)} variables to data_dictionary_clean.csv")

             Id    Référence Saisie           Type  \
2     ID102S2V9  PROJECTNAME      O          Label   
3            V5          SEX      O  Radio buttons   
4    ID102S2V10      BRTHDAT      O           Date   
5    ID102S2V11          AGE      O         Entier   
6    ID102S2V15      INVNAME      O          Texte   
7    ID102S2V12       SITEID      O          Texte   
8    ID102S2V14  SUBJECT_REF      O          Texte   
9    ID102S2V16      RFICDAT      O           Date   
12   ID102S2V18       INF1YN      O  Radio buttons   
13   ID102S2V19      INF1AYN      O  Radio buttons   
14   ID102S2V20      INF1BYN      O  Radio buttons   
15   ID102S2V21     INF1B1LS      O  Radio buttons   
16   ID102S2V22     INF1B2DT      O           Date   
17  ID102S2V182     INF1B5DT      O           Date   
18   ID102S2V24     INF1B3DT      O           Date   
19   ID102S2V25     INF1B4DT      O           Date   
20   ID102S2V27    INF1B5NUM      O           Réel   
21   ID102S2V34   INF1A1MESS

In [26]:
variables = df[df["Référence"].notna()].copy()

variables = variables[["Id", "Référence", "Saisie", "Type", "Format", "Intitulé"]]

print("Variables:", len(variables))
variables.head(20)

Variables: 201


,Id,Référence,Saisie,Type,Format,Intitulé
2,ID102S2V9,PROJECTNAME,O,Label,NaN,FH-Early
3,V5,SEX,O,Radio buttons,1 = Male \n2 = Female,Gender
4,ID102S2V10,BRTHDAT,O,Date,Précision = MM/YYYY\nValeurs par défaut = 01/0...,Date of birth
5,ID102S2V11,AGE,O,Entier,Taille : N\nMin = 18\nMax = 120,Age
6,ID102S2V15,INVNAME,O,Texte,Taille : N,Investigator
7,ID102S2V12,SITEID,O,Texte,Taille : N,Centre
8,ID102S2V14,SUBJECT_REF,O,Texte,Taille : N,Patient ID
9,ID102S2V16,RFICDAT,O,Date,Précision = JJ/MM/YYYY\nValeurs par défaut = 0...,Date of inclusion
12,ID102S2V18,INF1YN,O,Radio buttons,1 = Yes \n0 = No,Patient deceased at the time of inclusion
13,ID102S2V19,INF1AYN,O,Radio buttons,1 = Yes \n0 = No,"If so, did you verify that there was no object..."


In [27]:
variables["Type"].value_counts(dropna=False)

Type
Radio buttons     68
Réel              39
Entier            24
Date              21
Combo box         21
Texte             12
Label              8
Cases à cocher     6
Tableau            2
Name: count, dtype: int64

## Analyze the mock export ZIP
> Saved the unzipped files in the folder `export-example`
> 
> The `*_ANSI.csv` file are mock patient data, exported from the real eCRF system.
> 
> They are meant to show the exact structure that we will receive later (June)
> 
> Now we need to compare the dictionary variable names vs the actual exported CSV columns!

In [34]:
zip_files = list(Path(".").glob("*.zip"))

# extract contents of ZIP files
# with zipfile.ZipFile("24ACE-FH-Early_export_exemple_20260424.zip") as zip_ref:
#    zip_ref.extractall(".")
export_folder = Path("export-example")

for f in export_folder.glob("*_ANSI.csv"):
    df_csv = pd.read_csv(f, sep=None, engine="python", encoding="latin1", nrows=5)
    print(f.name, df_csv.shape)
    print(list(df_csv.columns))
    print()

ML_ANSI.csv (5, 272)
['STUDY_ID', 'COUNTRY_ID', 'EXTRACTION_DATE', 'SITE_ID', 'SUBJECT_ID', 'SUBJECT_REF', 'REF_SITEID', 'REF_NUMBER', 'MODULE_ML', 'CD1TXT', 'CD2LS', 'CD3NUM', 'CD3NUM_V', 'CD4NUM', 'CD4NUM_V', 'CD5NUM', 'CD5NUM_V', 'CD6NUM', 'CD6NUM_V', 'CD7NUM', 'CD7NUM_V', 'CD8YN', 'CD8AYN', 'MH1YN', 'MH1ANUM', 'MH1ANUM_V', 'MH2YN', 'MH2ANUM', 'MH2ANUM_V', 'MH23NUM', 'MH23NUM_V', 'MH2BLS', 'MH3YN', 'MH3ANUM', 'MH3ANUM_V', 'MH4YN', 'MH4ANUM', 'MH4ANUM_V', 'MH5YN', 'MH5AYN', 'MH5A1NUM', 'MH5A1NUM_V', 'MH5BYN', 'MH5B1NUM', 'MH5B1NUM_V', 'MH5CYN', 'MH5C1NUM', 'MH5C1NUM_V', 'MH5DYN', 'MH5D1NUM', 'MH5D1NUM_V', 'MH5EYN', 'MH5E1NUM', 'MH5E1NUM_V', 'MH5FYN', 'MH5F1NUM', 'MH5F1NUM_V', 'MH5GYN', 'MH5HYN', 'MH16H1NUM', 'MH16H1NUM_V', 'MH16IYN', 'MH16I1NUM', 'MH16I1NUM_V', 'MH16JYN', 'MH16J1NUM', 'MH16J1NUM_V', 'MH16KYN', 'MH16K1NUM', 'MH16K1NUM_V', 'MH16LYN', 'MH16L1NUM', 'MH16L1NUM_V', 'FH1NUM', 'FH1NUM_V', 'FH2DT_D', 'FH2DT_M', 'FH2DT_Y', 'FH2DT', 'FH3YN', 'FH4YN', 'FH5YN', 'FH6YN', 'FH7YN', 

## Inspect `INC_ANSI.csv`
### For each variable in the dictionary, what columns appear in the export CSV? 
> For instance: 
> * `BRTHDAT` in dictionary became `BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT` in .csv (as also mentioned in Alain email)

In [39]:
inc = pd.read_csv(export_folder / "INC_ANSI.csv", encoding="latin1", sep=";")
inc_cols = inc.columns.tolist()
print(inc.shape)

for var in ["SEX", "BRTHDAT", "AGE", "RFICDAT"]:
    matches = [c for c in inc_cols if c.startswith(var)]
    print(var, "->", matches)

# automate what above 
inc_matches = variables.copy()

inc_matches["export_columns"] = inc_matches["Référence"].apply(
    lambda var: [c for c in inc.columns if c.startswith(str(var))]
)

inc_matches[["Référence", "Type", "Intitulé", "export_columns"]].head(30)

(5, 73)
SEX -> ['SEX']
BRTHDAT -> ['BRTHDAT_D', 'BRTHDAT_M', 'BRTHDAT_Y', 'BRTHDAT']
AGE -> ['AGE', 'AGE_V']
RFICDAT -> ['RFICDAT_D', 'RFICDAT_M', 'RFICDAT_Y', 'RFICDAT']


,Référence,Type,Intitulé,export_columns
2,PROJECTNAME,Label,FH-Early,[]
3,SEX,Radio buttons,Gender,[SEX]
4,BRTHDAT,Date,Date of birth,"[BRTHDAT_D, BRTHDAT_M, BRTHDAT_Y, BRTHDAT]"
5,AGE,Entier,Age,"[AGE, AGE_V]"
6,INVNAME,Texte,Investigator,[INVNAME]
7,SITEID,Texte,Centre,[SITEID]
8,SUBJECT_REF,Texte,Patient ID,"[SUBJECT_REF, SUBJECT_REF_CL1]"
9,RFICDAT,Date,Date of inclusion,"[RFICDAT_D, RFICDAT_M, RFICDAT_Y, RFICDAT]"
12,INF1YN,Radio buttons,Patient deceased at the time of inclusion,[INF1YN]
13,INF1AYN,Radio buttons,"If so, did you verify that there was no object...",[INF1AYN]


In [40]:
inc[["SUBJECT_REF", "SUBJECT_REF_CL1"]].head()

,SUBJECT_REF,SUBJECT_REF_CL1
0,ZZOI-0001,ZZOI-0001
1,QGYD-0002,QGYD-0002
2,EJI-0003,EJI-0003
3,CH-0004,CH-0004
4,UHVT-0005,UHVT-0005


In [42]:
# check one radio buttons (for sex 1 is male and 2 female)
inc["SEX"].value_counts(dropna=False)

SEX
1.0    2
2.0    2
NaN    1
Name: count, dtype: int64

In [43]:
# check yes/no radio fields
inc["INF1YN"].value_counts(dropna=False)

INF1YN
0    5
Name: count, dtype: int64

In [45]:
# check date fields
# difference bewtween NaN and ND: Nan means missing data, ND means not determined
inc[["BRTHDAT_D", "BRTHDAT_M", "BRTHDAT_Y", "BRTHDAT"]].head()

,BRTHDAT_D,BRTHDAT_M,BRTHDAT_Y,BRTHDAT
0,NaN,7,1997.0,NaN
1,NaN,6,2007.0,NaN
2,NaN,NaN,NaN,NaN
3,NaN,4,1978.0,NaN
4,NaN,ND,1972.0,NaN


In [46]:
# check a complete date field
inc[["RFICDAT_D", "RFICDAT_M", "RFICDAT_Y", "RFICDAT"]].head()

,RFICDAT_D,RFICDAT_M,RFICDAT_Y,RFICDAT
0,20,2,2013,20/02/2013
1,17,2,1989,17/02/1989
2,22,9,1971,22/09/1971
3,5,5,2008,05/05/2008
4,27,12,2009,27/12/2009


In [47]:
# check numeric variables
inc[["AGE", "AGE_V"]].head()

,AGE,AGE_V
0,15,15
1,18,18
2,59,59
3,30,30
4,37,37


In [48]:
# check a numeric field that may have missing 
inc[["INF1B5NUM", "INF1B5NUM_V"]].head()

,INF1B5NUM,INF1B5NUM_V
0,NaN,NaN
1,1.2,1.2
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


## Inspect `ML_ANSI.csv`
### How to handle checkbox variables
> `INC_ANSI.csv` do not contain checkbox examples, but Alain warned that checkbox variables are transformed differently
> 
> A checkbox variable allow selecting multiple answer simultaneosly. Something as: Which sympthoms? 
> * [] fever
> * [] cough
> * [] headache
> In the email they said that checkbox variable become: `Variable_C1`, `Variable_C2`, `Variable_C3`...
> 
> And in `ML_ANSI.csv` investigation we saw: `FH7ALS_C1`, `FH7ALS_C2`, `FH7ALS_C3`...


In [51]:
# confirm theory from dictionary
variables[variables["Référence"] == "FH7ALS"]

,Id,Référence,Saisie,Type,Format,Intitulé
121,ID102S2V86,FH7ALS,O,Cases à cocher,1 = Mother \n2 = Father \n3 = child \n4 = sibl...,If yes: specify


In [50]:
ml = pd.read_csv(export_folder / "ML_ANSI.csv", encoding="latin1", sep=";")
checkbox_cols = [c for c in ml.columns if c.startswith("FH7ALS_C")]
print(checkbox_cols)
ml[checkbox_cols].head()

['FH7ALS_C1', 'FH7ALS_C2', 'FH7ALS_C3', 'FH7ALS_C4', 'FH7ALS_C5', 'FH7ALS_C6', 'FH7ALS_C7', 'FH7ALS_C8', 'FH7ALS_C9', 'FH7ALS_C10', 'FH7ALS_C11', 'FH7ALS_C12', 'FH7ALS_C13', 'FH7ALS_C14']


,FH7ALS_C1,FH7ALS_C2,FH7ALS_C3,FH7ALS_C4,FH7ALS_C5,FH7ALS_C6,FH7ALS_C7,FH7ALS_C8,FH7ALS_C9,FH7ALS_C10,FH7ALS_C11,FH7ALS_C12,FH7ALS_C13,FH7ALS_C14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
